<a href="https://colab.research.google.com/github/harinireddy2308/Nutriseeker/blob/main/NutriSeeker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install all required libraries
!pip install transformers torch gradio Pillow requests -q

import warnings
warnings.filterwarnings("ignore")

print("✅ All libraries installed!")

✅ All libraries installed!


In [ ]:
import pandas as pd
from google.colab import files

print("📂 Upload your ifct2017_compositions.csv file...")
uploaded = files.upload()  # A file picker will appear — select ifct2017_compositions.csv

df_real = pd.read_csv("ifct2017_compositions.csv")

print(f"\n✅ Real IFCT 2017 loaded!")
print(f"📊 Total foods: {len(df_real)}")
print(f"\n🔍 Column names:")
print(df_real.columns.tolist())
print(f"\n📋 First 3 rows:")
print(df_real.head(3))

📂 Upload your ifct2017_compositions.csv file...


Saving ifct2017_compositions.csv to ifct2017_compositions.csv

✅ Real IFCT 2017 loaded!
📊 Total foods: 542

🔍 Column names:
['code', 'name', 'scie', 'regn', 'water', 'water_e', 'protcnt', 'protcnt_e', 'ash', 'ash_e', 'fatce', 'fatce_e', 'fibtg', 'fibtg_e', 'fibins', 'fibins_e', 'fibsol', 'fibsol_e', 'choavldf', 'choavldf_e', 'enerc', 'enerc_e', 'dhbenzac34', 'dhbenzac34_e', 'hbenzal3', 'hbenzal3_e', 'pcathac', 'pcathac_e', 'vanlac', 'vanlac_e', 'gallac', 'gallac_e', 'cinmac', 'cinmac_e', 'coumaco', 'coumaco_e', 'coumacp', 'coumacp_e', 'caffac', 'caffac_e', 'chlrac', 'chlrac_e', 'ferac', 'ferac_e', 'apigen', 'apigen_e', 'apigen6cgls', 'apigen6cgls_e', 'apigen7onshps', 'apigen7onshps_e', 'luteol', 'luteol_e', 'kaemf', 'kaemf_e', 'querce', 'querce_e', 'querce3bdgls', 'querce3bdgls_e', 'querce3ortns', 'querce3ortns_e', 'querce3bgls', 'querce3bgls_e', 'isormt', 'isormt_e', 'myrct', 'myrct_e', 'rsvrtol', 'rsvrtol_e', 'hespt', 'hespt_e', 'narng', 'narng_e', 'hespd', 'hespd_e', 'daidzn', 'daid

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

print("⏳ Loading BLIP model... (first time takes 2-3 mins)")

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"✅ BLIP model loaded! Running on: {device}")

⏳ Loading BLIP model... (first time takes 2-3 mins)


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

✅ BLIP model loaded! Running on: cuda


In [ ]:
import requests

df = df_real.copy()

FOOD_KEYWORD_MAP = {
    # Grains
    "rice": "Rice, raw, milled",
    "biryani": "Rice, raw, milled",
    "poha": "Rice flakes",
    "puffed rice": "Rice puffed",
    "roti": "Wheat flour, atta",
    "chapati": "Wheat flour, atta",
    "paratha": "Wheat flour, atta",
    "bread": "Wheat flour, refined",
    "naan": "Wheat flour, refined",
    "upma": "Wheat, semolina",
    "semolina": "Wheat, semolina",
    "wheat": "Wheat, whole",
    "bajra": "Bajra",
    "ragi": "Finger millet",
    "millet": "Finger millet",
    "maize": "Maize, dry",
    "corn": "Maize, dry",
    "oat": "Oat",
    # Pulses
    "dal": "Red gram, dal",
    "lentil": "Lentil dal",
    "rajma": "Kidney bean",
    "chana": "Bengal gram, dal",
    "chickpea": "Bengal gram, dal",
    "moong": "Green gram, dal",
    "urad": "Black gram, dal",
    # Meat & Eggs
    "chicken": "Chicken, poultry, breast, skinless",
    "mutton": "Mutton, muscle",
    "egg": "Egg, poultry, whole, raw",
    "omelette": "Egg, poultry, whole, raw",
    "boiled egg": "Egg, poultry, whole, boiled",
    "fish": "Rohu",
    "prawn": "Prawn",
    # Dairy
    "milk": "Milk, whole, Cow",
    "curd": "Curd",
    "yogurt": "Curd",
    "paneer": "Paneer",
    "butter": "Butter",
    "ghee": "Ghee",
    # Vegetables
    "potato": "Potato, brown skin, big",
    "aloo": "Potato, brown skin, big",
    "tomato": "Tomato, ripe",
    "onion": "Onion, big",
    "spinach": "Spinach",
    "palak": "Spinach",
    "carrot": "Carrot",
    "cabbage": "Cabbage",
    "cauliflower": "Cauliflower",
    "brinjal": "Brinjal",
    # Fruits
    "banana": "Banana, ripe, robusta",
    "apple": "Apple",
    "mango": "Mango, ripe",
    "orange": "Orange",
    "grapes": "Grapes, black",
    "papaya": "Papaya, ripe",
    "watermelon": "Watermelon",
    # Nuts & Oils
    "cashew": "Cashewnut",
    "almond": "Almond",
    "peanut": "Groundnut",
    "groundnut": "Groundnut",
    "coconut": "Coconut, fresh",
}

def search_ifct(food_name):
    """Search real IFCT 2017 database"""
    food_name = food_name.lower().strip()

    # Step 1 — keyword map (both sides lowercased now ✅)
    for keyword, mapped_name in FOOD_KEYWORD_MAP.items():
        if keyword in food_name:
            result = df[df['name'].str.lower().str.contains(mapped_name.lower(), na=False)]
            if not result.empty:
                data = result.iloc[0]
                print(f"Keyword match: '{keyword}' → '{data['name']}'")
                return {
                    "source": "IFCT 2017",
                    "food": data['name'],
                    "calories": round(data['enerc'] / 4.184, 1),   # ✅ kJ → kcal
                    "protein": round(data['protcnt'], 1),
                    "carbs": round(data['choavldf'], 1),
                    "fat": round(data['fatce'], 1),
                    "fiber": round(data['fibtg'], 1)
                }

    # Step 2 — partial match
    result = df[df['name'].str.lower().str.contains(food_name, na=False)]
    if not result.empty:
        data = result.iloc[0]
        print(f"Partial match: '{data['name']}'")
        return {
            "source": "IFCT 2017",
            "food": data['name'],
            "calories": round(data['enerc'] / 4.184, 1),           # ✅ kJ → kcal fixed
            "protein": round(data['protcnt'], 1),
            "carbs": round(data['choavldf'], 1),
            "fat": round(data['fatce'], 1),
            "fiber": round(data['fibtg'], 1)
        }

    return None

def search_usda(food_name):
    """USDA fallback for non-Indian foods"""
    try:
        url = "https://api.nal.usda.gov/fdc/v1/foods/search"
        params = {"query": food_name, "api_key": "DEMO_KEY", "pageSize": 1}
        response = requests.get(url, params=params, timeout=5)
        data = response.json()
        if data.get('foods'):
            food = data['foods'][0]
            # ✅ Using correct USDA response field names
            nutrients = {n['nutrientName']: n['value'] for n in food.get('foodNutrients', [])}
            return {
                "source": "USDA",
                "food": food['description'],
                "calories": round(nutrients.get('Energy', 0), 1),
                "protein": round(nutrients.get('Protein', 0), 1),
                "carbs": round(nutrients.get('Carbohydrate, by difference', 0), 1),
                "fat": round(nutrients.get('Total lipid (fat)', 0), 1),
                "fiber": round(nutrients.get('Fiber, total dietary', 0), 1)
            }
    except:
        pass
    return None

def get_nutrition(food_name):
    """Master function — IFCT 2017 first, USDA fallback"""
    result = search_ifct(food_name)
    if result:
        print("✅ Found in IFCT 2017!")
        return result
    print("⚠️ Not in IFCT, trying USDA...")
    result = search_usda(food_name)
    if result:
        print("✅ Found in USDA!")
        return result
    return None

print("✅ Nutrition lookup ready with real IFCT 2017!")

# Test
for test_food in ["rice", "banana", "chicken", "potato", "egg", "milk"]:
    result = get_nutrition(test_food)
    if result:
        print(f"✅ {test_food} → {result['food']} | {result['calories']} kcal")
    else:
        print(f"❌ {test_food} → not found")


✅ Nutrition lookup ready with real IFCT 2017!
Keyword match: 'rice' → 'Rice, raw, milled'
✅ Found in IFCT 2017!
✅ rice → Rice, raw, milled | 356.4 kcal
Keyword match: 'banana' → 'Banana, ripe, robusta'
✅ Found in IFCT 2017!
✅ banana → Banana, ripe, robusta | 105.2 kcal
Keyword match: 'chicken' → 'Chicken, poultry, breast, skinless'
✅ Found in IFCT 2017!
✅ chicken → Chicken, poultry, breast, skinless | 168.3 kcal
Keyword match: 'potato' → 'Potato, brown skin, big'
✅ Found in IFCT 2017!
✅ potato → Potato, brown skin, big | 69.8 kcal
Keyword match: 'egg' → 'Egg, poultry, whole, raw'
✅ Found in IFCT 2017!
✅ egg → Egg, poultry, whole, raw | 134.8 kcal
Keyword match: 'milk' → 'Milk, whole, Cow'
✅ Found in IFCT 2017!
✅ milk → Milk, whole, Cow | 72.9 kcal


In [ ]:
# Check what rice-related foods exist in real IFCT
keywords_to_check = ["rice", "chicken", "banana", "potato", "wheat", "egg", "milk", "dal"]

for kw in keywords_to_check:
    matches = df[df['name'].str.lower().str.contains(kw, na=False)]['name'].tolist()
    print(f"\n🔍 '{kw}' matches:")
    for m in matches[:5]:  # show first 5 matches
        print(f"   → {m}")


🔍 'rice' matches:
   → Rice flakes
   → Rice puffed
   → Rice, raw, brown
   → Rice, parboiled, milled
   → Rice, raw, milled

🔍 'chicken' matches:
   → Chicken mushroom, fresh
   → Chicken, poultry, leg, skinless
   → Chicken, poultry, thigh, skinless
   → Chicken, poultry, breast, skinless
   → Chicken, poultry, wing, skinless

🔍 'banana' matches:
   → Banana, ripe, montham
   → Banana, ripe, poovam
   → Banana, ripe, red
   → Banana, ripe, robusta

🔍 'potato' matches:
   → Potato, brown skin, big
   → Potato, brown skin, small
   → Potato, red skin
   → Sweet potato, brown skin
   → Sweet potato, pink skin

🔍 'wheat' matches:
   → Wheat flour, refined
   → Wheat flour, atta
   → Wheat, whole
   → Wheat, bulgur
   → Wheat, semolina

🔍 'egg' matches:
   → Egg, poultry, whole, raw
   → Egg, poultry, white, raw
   → Egg, poultry, yolk, raw
   → Egg, poultry, whole, boiled
   → Egg, poultry, white, boiled

🔍 'milk' matches:
   → Milk, whole, Buffalo
   → Milk, whole, Cow
   → Milk fish


In [ ]:
# Test with a few foods
for test_food in ["rice", "banana", "chicken", "potato"]:
    result = get_nutrition(test_food)
    if result:
        print(f"✅ {test_food} → {result['food']} | {result['calories']} kcal")
    else:
        print(f"❌ {test_food} → not found")

Keyword match: 'rice' → 'Rice, raw, milled'
✅ Found in IFCT 2017!
✅ rice → Rice, raw, milled | 356.4 kcal
Keyword match: 'banana' → 'Banana, ripe, robusta'
✅ Found in IFCT 2017!
✅ banana → Banana, ripe, robusta | 105.2 kcal
Keyword match: 'chicken' → 'Chicken, poultry, breast, skinless'
✅ Found in IFCT 2017!
✅ chicken → Chicken, poultry, breast, skinless | 168.3 kcal
Keyword match: 'potato' → 'Potato, brown skin, big'
✅ Found in IFCT 2017!
✅ potato → Potato, brown skin, big | 69.8 kcal


In [ ]:
def identify_food(image):
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)

    # ✅ NO text parameter — captioning model needs image only
    inputs = processor(images=image, return_tensors="pt").to(device)

    output = model.generate(**inputs, max_new_tokens=50)
    full_description = processor.decode(output[0], skip_special_tokens=True)

    # Clean BLIP output
    food_name = full_description.lower()
    remove_phrases = [
        "a plate of", "a bowl of", "a dish of", "a serving of",
        "a photo of", "a picture of", "an image of", "i think this is",
        "this is", "the image shows", "there is"
    ]
    for phrase in remove_phrases:
        food_name = food_name.replace(phrase, "")

    for separator in [" and ", " with ", ",", " on a", " on the"]:
        food_name = food_name.split(separator)[0]

    food_name = food_name.strip()
    print(f"BLIP raw output   : {full_description}")
    print(f"Cleaned food name : {food_name}")

    return food_name, full_description

print("✅ Food identifier ready!")

✅ Food identifier ready!


In [ ]:
import gradio as gr

def analyze_meal(image, text_input):
    try:
        # ✅ Image is compulsory
        if image is None:
            return "❌ Please upload a food image! (text input alone is not supported)"

        # Detect food from image always
        food_name, full_description = identify_food(image)

        # If text also provided → override food name for better accuracy
        if text_input and text_input.strip():
            food_name = text_input.strip().lower()
            print(f"Text override: using '{food_name}' instead of BLIP output")
        else:
            print(f"Using BLIP detected name: '{food_name}'")

        # Lookup nutrition
        nutrition = get_nutrition(food_name)

        if nutrition:
            result = f"""
🍽️  MEAL ANALYSIS RESULTS
{'='*40}
📌 BLIP Detected  : {full_description}
🔍 Matched Food   : {nutrition['food']}
📚 Data Source    : {nutrition['source']}

{'='*40}
   NUTRITION INFO (per 100g)
{'='*40}
🔥 Calories  :  {nutrition['calories']} kcal
💪 Protein   :  {nutrition['protein']} g
🍞 Carbs     :  {nutrition['carbs']} g
🧈 Fat       :  {nutrition['fat']} g
🌾 Fiber     :  {nutrition['fiber']} g
{'='*40}
            """
        else:
            result = f"""
🍽️  MEAL ANALYSIS RESULTS
{'='*40}
📌 BLIP Detected : {full_description}
❌ Food not found in IFCT or USDA

💡 Tips:
  • Try typing the food name in the text box
  • Use a well-lit close-up image
  • Try a simpler food name (e.g. "rice" not "basmati rice")
{'='*40}
            """
        return result

    except Exception as e:
        return f"❌ Error: {str(e)}\nPlease re-run all cells from top."

# Launch Gradio
try:
    demo.close()
except:
    pass

demo = gr.Interface(
    fn=analyze_meal,
    inputs=[
        gr.Image(type="pil", label="📸 Upload Meal Photo (required)"),
        gr.Textbox(
            label="✏️ Food name — type to improve accuracy (optional)",
            placeholder="e.g. biryani, dal, paneer, pizza...",
            lines=1
        )
    ],
    outputs=gr.Textbox(label="📊 Nutrition Results", lines=18),
    title="🍽️ NutriSeeker — AI Meal Nutrition Analyzer",
    description="📸 Image is required. Typing the food name is optional but improves accuracy!",
    theme=gr.themes.Soft()
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2fbac4271f6d07c864.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
